In [28]:
import xarray as xr
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from pyresample import geometry
from pyresample.kd_tree import resample_gauss


In [29]:
ds_grid=xr.open_dataset("../../ensemble_reduced/0/yelmo2D_reduced.nc")


In [ ]:
u_x = xr.open_dataset("./greenland_vel_mosaic250_vx_v1.tif", engine="rasterio")
u_y = xr.open_dataset("./greenland_vel_mosaic250_vy_v1.tif", engine="rasterio")
err_ux = xr.open_dataset("./greenland_vel_mosaic250_ex_v1.tif", engine="rasterio")
err_uy = xr.open_dataset("./greenland_vel_mosaic250_ey_v1.tif", engine="rasterio")

source = "Joughin, I., Smith, B. and Howat, I.: A complete map of Greenland ice velocity derived from satellite data collected over 20 years. Journal of Glaciology, 64(243), 1-11, doi:10.1017/jog.2017.73, 2018"

ds = xr.Dataset(
    data_vars={
        "u_x": (("yc","xc"), np.array(u_x.band_data.squeeze('band').drop_vars('band'))),
        "u_y": (("yc","xc"), np.array(u_y.band_data.squeeze('band').drop_vars('band'))),
        "err_ux": (("yc","xc"), np.array(err_ux.band_data.squeeze('band').drop_vars('band'))),
        "err_uy": (("yc","xc"), np.array(err_uy.band_data.squeeze('band').drop_vars('band')))},
    coords={
        "xc": np.array(u_x.x*1e-3),
        "yc": np.array(u_x.y*1e-3)},
    attrs={"source":(source),
           "units":("m/yr")})

ds.xc.attrs = {'units': 'km', 'long_name': 'x-coordinate'}
ds.yc.attrs = {'units': 'km', 'long_name': 'y-coordinate'}
ds = ds.sortby("yc", ascending=True)

ds=ds.fillna(0)
ds.to_netcdf("./output/vel_J18.nc")

In [ ]:
# Projection of the original dataset and the target grid is EPSG:3413 or NSIDC Sea Ice Polar Stereographic North, BUT in km
proj_str = "+proj=stere +lat_0=90 +lat_ts=70 +lon_0=-45 +k=1 +x_0=0 +y_0=0 +datum=WGS84 +units=km +no_defs"

def get_area_def_km(ds, name):
    res_x = abs(ds.xc[1] - ds.xc[0]).values
    res_y = abs(ds.yc[1] - ds.yc[0]).values

    extent = (
        ds.xc.min().values - res_x/2,
        ds.yc.min().values - res_y/2,
        ds.xc.max().values + res_x/2,
        ds.yc.max().values + res_y/2
    )
    
    return geometry.AreaDefinition(
        name, name, 'nps_area_km',
        proj_str,
        len(ds.xc), len(ds.yc),
        extent
    )

area_orig = get_area_def_km(ds, "high_res_250m")
area_target = get_area_def_km(ds_grid, "low_res_8km")


In [ ]:
# 2. Regrid with error propagation
R = 8.0 

def process_stats(da, area_in, area_out, radius):
    # mean E[X]
    mean = resample_gauss(area_in, da.values, area_out, 
                          radius_of_influence=radius, sigmas=radius/2, fill_value=np.nan)
    # mean of squares: E[X^2]
    mean_sq = resample_gauss(area_in, da.values**2, area_out, 
                             radius_of_influence=radius, sigmas=radius/2, fill_value=np.nan)
    
    # standard deviation sub-grid: sqrt( E[X^2] - (E[X])^2 )
    std_sub = np.sqrt(np.maximum(0, mean_sq - mean**2))
    return mean, std_sub

u_x_8km, std_ux = process_stats(ds.u_x, area_orig, area_target, R)
u_y_8km, std_uy = process_stats(ds.u_y, area_orig, area_target, R)

# average of instrumental errors (only mean)
err_ux_inst = resample_gauss(area_orig, ds.err_ux.values, area_target, 
                             radius_of_influence=R, sigmas=R/2, fill_value=np.nan)
err_uy_inst = resample_gauss(area_orig, ds.err_uy.values, area_target, 
                             radius_of_influence=R, sigmas=R/2, fill_value=np.nan)

# 3. Total error
err_ux_total = np.sqrt(err_ux_inst**2 + std_ux**2)
err_uy_total = np.sqrt(err_uy_inst**2 + std_uy**2)


In [ ]:
ds_final = xr.Dataset(
    data_vars={
        "uxy_srf": (("yc", "xc"),np.sqrt(u_x_8km**2 + u_y_8km**2)),
        "uxy_err": (("yc", "xc"),np.sqrt(err_ux_total**2 + err_uy_total**2)),
        "uxy_err_cor": (("yc", "xc"),np.sqrt(err_ux_total**2 + err_uy_total**2 + (0.03*u_x_8km)**2 + (0.03*u_y_8km)**2)),
        "u_x": (("yc", "xc"), u_x_8km),
        "u_y": (("yc", "xc"), u_y_8km),
        "ux_err_inst": (("yc", "xc"), err_ux_inst),
        "uy_err_inst": (("yc", "xc"), err_uy_inst),
        "ux_std": (("yc", "xc"), std_ux),
        "uy_std": (("yc", "xc"), std_uy),
        "ux_err": (("yc", "xc"), err_ux_total),
        "uy_err": (("yc", "xc"), err_uy_total)
    },
    coords={
        "xc": ds_grid.xc,
        "yc": ds_grid.yc
    }
)
ds_final.to_netcdf("./output/vel_J18_8km.nc")